In [ ]:
# set jax defaults first?

import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import pickle
from jax import grad, vmap, tree_util
import jax.numpy as jnp
from jaxopt import GaussNewton
from jaxopt import LevenbergMarquardt
import pylab as plt
from numpy import random
import pandas as pd
import numpy as np

In [ ]:
#load stellar parameters
DATESTR = "2026-07-20"

stars = pd.read_parquet(f"parent_stars10k_{DATESTR}.parquet")
print("num of stars in parquet:", stars.shape[0])

spectra = pd.read_parquet(f"spectra_iter1_2026-08-07_plot_test.parquet")
print("num of spectra in parquet:", spectra.shape[0])

print(stars.shape, spectra.shape)


with open(f'2026-08-07_plot_test_iter0_synthdata.pkl','rb') as f:
    pkl_data = pickle.load(f)
print(len(pkl_data))

synth, data, __, weights, loglam = pkl_data
print(synth.shape, data.shape, weights.shape, loglam.shape)


In [ ]:
def get_delta_log_lam(loglambdas):
    return np.median(loglambdas[1:] - loglambdas[:-1])

In [ ]:
# define one-d Gaussian

HALFLNTWOPI = 0.5 * jnp.log(2. * jnp.pi)

def ln_gaussian(xs, mu, sigma):
    return -0.5 * (xs - mu) ** 2 / sigma ** 2 - jnp.log(jnp.abs(sigma)) - HALFLNTWOPI

def gaussian(xs, mu, sigma):
    return jnp.exp(ln_gaussian(xs, mu, sigma))

In [ ]:
# define mixture of two Gaussians, with log amplitudes

def ln_two_gaussians(xs, pars):
    lnamp1, mu1, sigma1, lnamp2, mu2, sigma2 = pars 
    return jnp.logaddexp(lnamp1 + ln_gaussian(xs, mu1, sigma1),
                         lnamp2 + ln_gaussian(xs, mu2, sigma2))

def two_gaussians(xs, pars):
    return jnp.exp(ln_two_gaussians(xs, pars))

In [ ]:
# root-finding with Jax -- I think I can do everything now with root finding

def find_root_step(f, x):
    return x - f(x) / grad(f)(x)

def find_root(f, x0, maxiter=5):
    x = 1. * x0
    for iter in range(maxiter):
        x = find_root_step(f, x)
    return x

In [ ]:
# use root finding to find the LHS and RHS of the line

def max_fwhm_and_double_peaked(pars, tiny=1.e-6):
    ff = lambda x: grad(ln_two_gaussians)(x, pars)
    root1 = find_root(ff, pars[1])
    root2 = find_root(ff, pars[4])
    hm = 0.5 * jnp.maximum(two_gaussians(root1, pars), two_gaussians(root2, pars))
    ff = lambda x: two_gaussians(x, pars) - hm
    leftside = find_root(ff, jnp.minimum(pars[1], pars[4]) - jnp.maximum(pars[2], pars[5]))
    rightside = find_root(ff, jnp.maximum(pars[1], pars[4]) + jnp.maximum(pars[2], pars[5]))
    return 2. * hm, leftside, rightside, jnp.abs(root1 - root2) > tiny

In [ ]:
def halpha_window(log_lambda, flxs, ivrs, synth_spectra, line, window = 15):
    #find the pixel closest to Halpha and return the fluxes, ivars, synth spectra, and lambdas values from that pixel +- window

    print(flxs.shape, ivrs.shape, synth_spectra.shape)
    lam = 10 ** log_lambda

    i = jnp.argmin(jnp.abs(lam - line)) #this gets the pixel thats closest to Halpha

    return flxs[:, i - window: i + window], ivrs[:, i - window: i + window], synth_spectra[:, i - window: i + window], lam[i - window : i + window]

In [ ]:
H_ALPHA, H_BETA = 6564.614, 4862.721 #Reference: from classic.sdss.org

In [ ]:
Ha_data, Ha_weights, Ha_synth, Ha_lam = halpha_window(loglam, data, weights, synth, H_ALPHA)
print(Ha_data.shape, Ha_weights.shape, Ha_synth.shape, Ha_lam.shape)

In [ ]:
def get_param_vals(ew, line = H_ALPHA):
    
    n = ew.shape[0]
    lnamp1 = lnamp2 = jnp.log(ew) - jnp.log(2)  
    sigma1 = sigma2 = jnp.full(n, 4.0)     
    center1 = jnp.full(n, line - 2)               
    center2 = jnp.full(n, line + 2)            

    vals = jnp.stack([lnamp1, center1, sigma1, lnamp2, center2, sigma2], axis=1)
    return vals   # shoudl be shape (N, 6)


def get_one_param_vals(ew, line = H_ALPHA):
    
    lnamp1 = lnamp2 = jnp.log(ew) - jnp.log(2)    
    sigma1 = sigma2 = 4.0
    center1 = line - 2           
    center2 = line + 2               
    vals = lnamp1, center1, sigma1, lnamp2, center2, sigma2
    return vals   

def fit_one_spectrum(lam, resid_i, ivar_i, values_i):
    chi = lambda pp: jnp.sqrt(ivar_i) * (resid_i - two_gaussians(lam, pp))
    LM = LevenbergMarquardt(residual_fun=chi)
    sol = LM.run(values_i)
    return sol.params

In [ ]:
#check?
ews = spectra["nana_Halpha_EW"].to_numpy()
all_pars = get_param_vals(ews) 
print(all_pars.shape)  
print(all_pars[0].shape)        
print(all_pars[1].shape)
print(all_pars[2].shape)

In [ ]:
resid = Ha_data - Ha_synth
print(resid.shape)

#Plotting example
print(ews.shape)
exi = 10002
ew_example = ews[exi]
all_params_example = get_one_param_vals(ew_example)
Ha_data, Ha_weights, Ha_synth, Ha_lam = halpha_window(loglam, data, weights, synth, H_ALPHA)


top, left, right, double_peak = max_fwhm_and_double_peaked(all_params_example)

# check FWHM code
plt.plot(Ha_lam, resid[exi], "k.") 
plt.plot(Ha_lam, two_gaussians(Ha_lam, all_params_example), "r-")
top, left, right, double_peak = max_fwhm_and_double_peaked(all_params_example)
print(top, right - left, double_peak)
plt.plot([left, right], [0.5 * top, 0.5 * top], "k-", lw=1)
plt.title("Example spectrum")
print(double_peak)

In [ ]:
#add columns to spectra df

spectra["fit_Halpha_EW"] = 0.
spectra["fit_Halpha_EW_err"] = jnp.inf
spectra["fit_Halpha_FWHM"] = 0.
spectra["fit_Halpha_dp"] = False
spectra["fit_Halpha_centroid"] = 0.

print(spectra.columns.to_list())


In [ ]:
def get_fit_ew(spectra_df, lambda_window, synth_window, data_window, ivars_window, line = H_ALPHA, LN10 = np.log(10.)):

    #get ew for spectra df and get the param values for eaxh
    ews = spectra_df["nana_Halpha_EW"].to_numpy()

    #
    fit_one_batched = vmap(fit_one_spectrum, in_axes=(None, 0, 0, 0))
    resid_window = data_window - synth_window
    all_params = fit_one_batched(lambda_window, resid_window, ivars_window, get_param_vals(ews))
    
    #fit centroid calcualtion
    a1, a2 = jnp.exp(all_params[:, 0]), jnp.exp(all_params[:, 3])
    mu1, mu2 = all_params[:, 1], all_params[:, 4]
    fit_centroid = (a1 * mu1 + a2 * mu2) / (a1 + a2)

    
    #get the max fwhm and double peak for all spectra
    max_fwhm_and_double_peaked_ALL = vmap(max_fwhm_and_double_peaked)
    tops, lefts, rights, double_peaks = max_fwhm_and_double_peaked_ALL(all_params)

    #compute model for all spectra
    two_gaussians_ALL = vmap(two_gaussians, in_axes=(None, 0))  
    fit_model = two_gaussians_ALL(lambda_window, all_params)  
    print(fit_model.shape)

    #caluclate the fit_ew and it's error
    delta_log_lambda_pixel = get_delta_log_lam(jnp.log10(lambda_window))
    integration_weight = delta_log_lambda_pixel * LN10 * lambda_window

    fit_ews = jnp.sum(fit_model * integration_weight[None, :], axis=1)
    fit_ew_errs = jnp.sqrt(jnp.sum(integration_weight[None, :] ** 2 / ivars_window, axis=1))

    
        spectra_df["fit_Halpha_EW"] = fit_ews
    spectra_df["fit_Halpha_EW_err"] = fit_ew_errs
    spectra_df["fit_Halpha_FWHM"] = rights - lefts
    spectra_df["fit_Halpha_dp"] = double_peaks
    spectra_df["fit_Halpha_centroid"] = fit_centroid

    with open(f"2026-08-17_fit_model_data.pkl", "wb") as file:
        pickle.dump((fit_model, fit_ews), file) #for the pretty spectrum fitting

    return
    


In [ ]:
Ha_data, Ha_weights, Ha_synth, Ha_lam = halpha_window(loglam, data, weights, synth, H_ALPHA)
get_fit_ew(spectra, Ha_lam,Ha_synth, Ha_data,  Ha_weights)

In [ ]:
temps = spectra["Teff_fit"].to_numpy()
fit_ew = spectra["fit_Halpha_EW"].to_numpy()
nana_ew = spectra["nana_Halpha_EW"].to_numpy()
nana_ew_err = spectra["nana_Halpha_EW_err"].to_numpy()

m2 = spectra["nana_Halpha_M2"].to_numpy()
nana_linewidth = np.sqrt(m2/nana_ew)
fit_FWHM = spectra["fit_Halpha_FWHM"].to_numpy()
m1 = spectra["nana_Halpha_M1"].to_numpy()
fit_centroid = spectra["fit_Halpha_centroid"].to_numpy()
fit_FWHM= spectra["fit_Halpha_FWHM"].to_numpy()

In [ ]:
print(f"Number of nans in fit FWHM: {np.sum(np.isnan(fit_FWHM))} out of {len(fit_FWHM)}")
print(f"Number of nans in nana linewidth: {np.sum(np.isnan(nana_linewidth))} out of {len(nana_linewidth)}")
mask_ew0 = (nana_ew <= 0)
print(f"Number of nana ew <= 0: {np.sum(mask_ew0)}")
print(np.where(mask_ew0))
print(np.where(np.isnan(fit_FWHM)))

In [ ]:
### Scatter plot of nana linewidth and fit FWHM

bof = spectra["nana_bof"].to_numpy()

fit_FWHM = jnp.abs(fit_FWHM)
good = good = (bof < 1.8) & (nana_ew_err < 1.0)
be = (nana_ew > 1)
be_good = good & be

plt.figure(figsize = (8,6))
plt.title("Be and good, nana linewidth vs fit FWHM")
plt.xlabel("Nana linewidth")
plt.ylabel("fit FWHM")
plt.scatter(nana_linewidth[be_good], fit_FWHM[be_good], alpha = 0.4,s = 8, lw=0, color = "k")   # , c = temps)
#sc = plt.scatter(nana_linewidth, fit_FWHM, alpha = 0.4,s = 3, lw=0, c = temps)
#colors = sc.to_rgba(temps)
#plt.errorbar(nana_ew, fit_ew, xerr = nana_ew_err, fmt = "none", ecolor = colors, markersize = 2, alpha = 0.3, zorder = 5)
plt.axvline(0, color = "blue", alpha = 0.4)
plt.axhline(0, color = "blue", alpha = 0.4)
plt.xlim(0,20)
plt.ylim(0, 20)
# plt.xlim(-5,20)
# plt.ylim(-15,10)
# plt.xlim(35,45)
# plt.ylim(-25,20)
#plt.colorbar()
plt.show()

In [ ]:
big_fwhm = (fit_FWHM > 100) & be_good
print("big fwhm indices")
print(np.where(big_fwhm)[0])
print("fwhm value:", fit_FWHM[22622])

In [ ]:
small_fwhm = (fit_FWHM == 0) & be_good
print(np.where(small_fwhm)[0])
print("fwhm value:", fit_FWHM[20160])

In [ ]:
good = good = (bof < 1.8) & (nana_ew_err < 1.0)
be = (nana_ew > 1)
be_good = good & be
plt.figure(figsize = (8,6))
plt.title("Good & Be, nana ew vs fit ew")
plt.xlabel("Nana ew")
plt.ylabel("fit ew")
#plt.scatter(nana_ew[be_good], fit_ew[be_good], alpha = 0.4,s = 3, lw=0, c = temps)
sc = plt.scatter(nana_ew[be_good], fit_ew[be_good], alpha = 0.4,s = 3, lw=0, c = temps[be_good])
colors = sc.to_rgba(temps[be_good])
plt.errorbar(nana_ew[be_good], fit_ew[be_good], xerr = nana_ew_err[be_good], fmt = "none", ecolor = colors, markersize = 2, alpha = 0.3, zorder = 5)
plt.axvline(0, color = "blue", alpha = 0.4)
plt.axhline(0, color = "blue", alpha = 0.4)
# plt.xlim(-5,20)
# plt.ylim(-15,10)
# plt.xlim(35,45)
# plt.ylim(-25,20)
plt.colorbar()
plt.show()

In [ ]:
#plot case where abs(nana_EW - fit_EW) > 1.5 

mask = np.abs(nana_ew - fit_ew) > 1.5
mask_indices = np.where(mask & be_good)
print(mask_indices)
print(len(mask_indices[0]))

plt.figure(figsize = (8,6))
plt.title("abs(nana_EW - fit_EW) > 1.5 , nana ew vs fit ew")
plt.xlabel("Nana ew")
plt.ylabel("fit ew")
#plt.scatter(nana_ew[be_good], fit_ew[be_good], alpha = 0.4,s = 3, lw=0, c = temps)
sc = plt.scatter(nana_ew[mask], fit_ew[mask], alpha = 0.4,s = 3, lw=0, c = temps[mask])
colors = sc.to_rgba(temps[mask])
#plt.errorbar(nana_ew[mask], fit_ew[mask], xerr = nana_ew_err[mask], fmt = "none", ecolor = colors, markersize = 2, alpha = 0.3, zorder = 5)
plt.axvline(0, color = "blue", alpha = 0.4)
plt.axhline(0, color = "blue", alpha = 0.4)
# plt.xlim(-5,20)
# plt.ylim(-15,10)
# plt.xlim(35,45)
# plt.ylim(-25,20)
plt.colorbar()
plt.show()

In [ ]:
### nana centroid vs fit centroid scatter plot 
### Scatter plot of nana linewidth and fit FWHM

good = good = (bof < 1.8) & (nana_ew_err < 1.0)
be = (nana_ew > 1)
be_good = good & be

nana_centroid = (m1/nana_ew)

plt.figure(figsize = (8,6))
plt.title("Be and good, nana centroid vs fit centroid")
plt.xlabel("Nana centroid")
plt.ylabel("fit centroid")
plt.plot( [-100, 100], [-100, 100], lw=1, alpha=0.5, color = "k", zorder = -10)
plt.scatter(nana_centroid[be_good], (fit_centroid - H_ALPHA)[be_good], alpha = 0.4,s = 8, lw=0.2, c = nana_ew[be_good], edgecolors = "black", cmap="viridis_r")
#sc = plt.scatter(nana_linewidth, fit_FWHM, alpha = 0.4,s = 3, lw=0, c = temps)
#colors = sc.to_rgba(temps)
#plt.errorbar(nana_ew, fit_ew, xerr = nana_ew_err, fmt = "none", ecolor = colors, markersize = 2, alpha = 0.3, zorder = 5)
# plt.axvline(0, color = "blue", alpha = 0.4)
# plt.axhline(0, color = "blue", alpha = 0.4)
plt.xlim(-30, 30)
plt.ylim(-100,100)
plt.colorbar()
plt.show()

In [ ]:
objects = (nana_centroid < 5) & ((fit_centroid - H_ALPHA) > 10) & (be_good)
print(np.where(objects))